In [ ]:
import geopandas as gpd
import numpy as np
import pandas as pd
import xarray as xr

In [7]:
# Define grid resolution
resolution = 50

# Setup grid parameters with spatial origin and grid size
x0, y0 = 585000, 2350004
x1, y1 = 632500, 2397504

# Boundary point where ocean-driver forcing is setup
x, y = 601900.8911782742, 2356315.456879479
gdf_boundary_points = gpd.GeoDataFrame(geometry=gpd.points_from_xy([x], [y]), crs=6634)

# Grid configuration dictionary
fixed_parameters = {
    "path_to_dem_tif": "inputs/dem.tif",
    "dem_tif_var_name": "elevtn",
    "path_to_rgh_tif": "inputs/sfincs.man",
    "path_to_outflow_shp": "inputs/outflow_boundaries.shp",
    "path_to_waterlevel_shp": "inputs/inflow_boundaries.shp",
    "x0": x0,  # Origin location of the cell edge (x0)
    "y0": y0,  # Origin location of the cell edge (y0)
    "dx": resolution,  # Grid size in x-direction
    "dy": resolution,  # Grid size in y-direction
    "mmax": (x1 - x0) // resolution,  # Number of grid cells in x-direction
    "nmax": (y1 - y0) // resolution,  # Number of grid cells in y-direction
    "rotation": 0,  # Rotation angle in degrees (anti-clockwise from east)
    "epsg": 6634,  # Coordinate reference system (CRS) EPSG code
    "path_to_mask": "inputs/mask.shp",
    "nr_subgrid_pixels": 1,
    "gdf_boundary_points": gdf_boundary_points,
}

In [8]:
# Creating time chunks as individual cases
K = 5

In [ ]:
# Waterlevels
waterlevels_forcing = pd.read_pickle("inputs/waterlevels.pkl")

time_chunks = np.array_split(waterlevels_forcing.index.sort_values().unique(), K)
time_bounds = [(edge[0], edge[-1]) for edge in time_chunks]

waterlevels_forcing = np.array_split(waterlevels_forcing, K)

slowly_waterlevel = [
    pd.DataFrame(i["Msetup"]).set_axis([0], axis=1) for i in waterlevels_forcing
]
quickly_waterlevel = [
    pd.DataFrame(i["Hig"]).set_axis([0], axis=1) for i in waterlevels_forcing
]

In [ ]:
# Precipitation
precipitation_forcing = xr.open_dataset("inputs/precipitation.nc")
precipitation_forcing = precipitation_forcing.rio.write_crs("epsg:4326")
precipitation_forcing = precipitation_forcing.sortby("time")

precipitation_forcing = [
    precipitation_forcing.sel(time=slice(str(start), str(end)))
    for start, end in time_bounds
]

In [ ]:
metamodel_parameters = {
    "precipitation_forcing": precipitation_forcing,
    "slowly_waterlevel_forcing": slowly_waterlevel,
    "quickly_waterlevel_forcing": quickly_waterlevel,
}

In [ ]:
from bluemath_tk.wrappers.sfincs.sfincs_wrapper import SfincsModelWrapper

sfincs_model = SfincsModelWrapper(
    templates_dir="templates",
    metamodel_parameters=metamodel_parameters,
    fixed_parameters=fixed_parameters,
    output_dir="CASES",
)

2025-10-02 03:40:51,291 - SfincsAlbaModelWrapper - WARNING - Parameter precipitation_forcing is not in the default_parameters
2025-10-02 03:40:51,293 - SfincsAlbaModelWrapper - WARNING - Parameter slowly_waterlevel_forcing is not in the default_parameters
2025-10-02 03:40:51,339 - SfincsAlbaModelWrapper - WARNING - Parameter quickly_waterlevel_forcing is not in the default_parameters


In [8]:
sfincs_model.build_cases()

Model dir already exists and files might be overwritten: /home/ibbisin0/globus/ricondal/Bluemath_Github/BlueMath/methods/dynamical_downscaling/SfincsExample/Alba_CASES/0000/gis.


Model dir already exists and files might be overwritten: /home/ibbisin0/globus/ricondal/Bluemath_Github/BlueMath/methods/dynamical_downscaling/SfincsExample/Alba_CASES/0001/gis.
Model dir already exists and files might be overwritten: /home/ibbisin0/globus/ricondal/Bluemath_Github/BlueMath/methods/dynamical_downscaling/SfincsExample/Alba_CASES/0002/gis.
Model dir already exists and files might be overwritten: /home/ibbisin0/globus/ricondal/Bluemath_Github/BlueMath/methods/dynamical_downscaling/SfincsExample/Alba_CASES/0003/gis.
Model dir already exists and files might be overwritten: /home/ibbisin0/globus/ricondal/Bluemath_Github/BlueMath/methods/dynamical_downscaling/SfincsExample/Alba_CASES/0004/gis.


In [ ]:
sfincs_model.run_cases(launcher="docker")

2025-10-02 03:18:10,417 - SfincsAlbaModelWrapper - ERROR - Error running command: docker run --rm -v .:/case_dir -w /case_dir deltares/sfincs-cpu
2025-10-02 03:18:10,418 - SfincsAlbaModelWrapper - ERROR - Error: Command 'docker run --rm -v .:/case_dir -w /case_dir deltares/sfincs-cpu' returned non-zero exit status 2.
2025-10-02 03:18:13,839 - SfincsAlbaModelWrapper - ERROR - Error running command: docker run --rm -v .:/case_dir -w /case_dir deltares/sfincs-cpu
2025-10-02 03:18:13,840 - SfincsAlbaModelWrapper - ERROR - Error: Command 'docker run --rm -v .:/case_dir -w /case_dir deltares/sfincs-cpu' returned non-zero exit status 2.
2025-10-02 03:18:16,976 - SfincsAlbaModelWrapper - ERROR - Error running command: docker run --rm -v .:/case_dir -w /case_dir deltares/sfincs-cpu
2025-10-02 03:18:16,976 - SfincsAlbaModelWrapper - ERROR - Error: Command 'docker run --rm -v .:/case_dir -w /case_dir deltares/sfincs-cpu' returned non-zero exit status 2.
2025-10-02 03:18:20,365 - SfincsAlbaModelWr